In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import multilabel_confusion_matrix, ConfusionMatrixDisplay, f1_score, hamming_loss
from sklearn.dummy import DummyClassifier
from statsmodels.stats.contingency_tables import mcnemar
import csv
import pandas as pd

In [ ]:
# Grab the results from the zero-shot csv file and store them to be used for confusion matrix creation
y_pred = []
y_true = []
#grab results from csv
with open('category_model_results_train_zero_think.csv', mode='r', newline='') as file:
    reader = csv.reader(file)
    header = next(reader)
    for row in reader:
        predicted = row[1:17]
        truth = row[17:]
        y_pred.append(predicted)
        y_true.append(truth)
        
y_pred = np.array(y_pred)
y_true = np.array(y_true)
print(y_pred)
print(y_true)

In [ ]:
# Grab the results from the few-shot csv file and store them to be used for confusion matrix creation
y_pred_few = []
y_true_few = []
#grab results from csv
with open('category_model_results_train_few_think.csv', mode='r', newline='') as file:
    reader = csv.reader(file)
    header = next(reader)
    for row in reader:
        predicted = row[1:17]
        truth = row[17:]
        y_pred_few.append(predicted)
        y_true_few.append(truth)
        
y_pred_few = np.array(y_pred_few)
y_true = np.array(y_true_few)
print(y_pred_few)
print(y_true_few)

In [ ]:
#list out categories in numpy array
categories = ['Result Category 1', 'Result Category 2', 'Result Category 3', 'Result Category 4', 'Result Category 5', 'Result Category 6', 'Result Category 7', 'Result Category 8', 'Result Category 9', 'Result Category 10', 'Result Category 11', 'Result Category 12', 'Result Category 13', 'Result Category 14', 'Result Category 15', 'Result Category 16']
category_indices = np.arange(len(categories))
#read in the predicted data and turn into binary, same order as the categories
samples = (y_pred == 'YES').astype(int)


#one hot encode the ground truth so we can put ground truth and predicted values into confusion matrix
ground_truth = (y_true == 'YES').astype(int)

print(samples)
print(ground_truth)

In [ ]:
#list out categories in numpy array
categories = ['Result Category 1', 'Result Category 2', 'Result Category 3', 'Result Category 4', 'Result Category 5', 'Result Category 6', 'Result Category 7', 'Result Category 8', 'Result Category 9', 'Result Category 10', 'Result Category 11', 'Result Category 12', 'Result Category 13', 'Result Category 14', 'Result Category 15', 'Result Category 16']
category_indices = np.arange(len(categories))
#read in the predicted data and turn into binary, same order as the categories
samples_few = (y_pred_few == 'YES').astype(int)


#one hot encode the ground truth so we can put ground truth and predicted values into confusion matrix
ground_truth_few = (y_true_few == 'YES').astype(int)

print(samples_few)
print(ground_truth_few)

In [ ]:
#multilabel confusion matrix
#https://scikit-learn.org/stable/modules/generated/sklearn.metrics.multilabel_confusion_matrix.html
mcm = multilabel_confusion_matrix(ground_truth, samples)

#display results for each category
labels = len(categories)
cols = 4
rows = 4

#this source helped with the displays for this quite a bit
#https://stackoverflow.com/questions/62722416/plot-confusion-matrix-for-multilabel-classifcation-python#:~:text=from%20sklearn.metrics%20import%20confusion_matrix%2C%20ConfusionMatrixDisplay%20import%20matplotlib.pyplot,disp.ax_.set_ylabel(%27%27)%20disp.im_.colorbar.remove()%20plt.subplots_adjust(wspace=0.10%2C%20hspace=0.1)%20f.colorbar(disp.im_%2C%20ax=axes)%20plt.show()
fig, axes = plt.subplots(nrows=rows, ncols=cols, figsize=(15, rows * 4))
for i, (matrix, ax) in enumerate(zip(mcm, axes.flatten())):
    disp = ConfusionMatrixDisplay(confusion_matrix=matrix)
    disp.plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(f"Label: {categories[i]}")

plt.tight_layout()
plt.savefig('train_zero_shot_think_cm.png', dpi=300, bbox_inches='tight')
plt.show()

#get F1 score for each cateogry
score = f1_score(ground_truth, samples, average=None)
print(score)


In [ ]:
#multilabel confusion matrix
#https://scikit-learn.org/stable/modules/generated/sklearn.metrics.multilabel_confusion_matrix.html
mcm = multilabel_confusion_matrix(ground_truth_few, samples_few)

#display results for each category
labels = len(categories)
cols = 4
rows = 4

#this source helped with the displays for this quite a bit
#https://stackoverflow.com/questions/62722416/plot-confusion-matrix-for-multilabel-classifcation-python#:~:text=from%20sklearn.metrics%20import%20confusion_matrix%2C%20ConfusionMatrixDisplay%20import%20matplotlib.pyplot,disp.ax_.set_ylabel(%27%27)%20disp.im_.colorbar.remove()%20plt.subplots_adjust(wspace=0.10%2C%20hspace=0.1)%20f.colorbar(disp.im_%2C%20ax=axes)%20plt.show()
fig, axes = plt.subplots(nrows=rows, ncols=cols, figsize=(15, rows * 4))
for i, (matrix, ax) in enumerate(zip(mcm, axes.flatten())):
    disp = ConfusionMatrixDisplay(confusion_matrix=matrix)
    disp.plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(f"Label: {categories[i]}")

plt.tight_layout()
plt.savefig('train_few_shot_think_cm.png', dpi=300, bbox_inches='tight')
plt.show()

#get F1 score for each cateogry
score = f1_score(ground_truth_few, samples_few, average=None)
print(score)


In [ ]:
#This link helped us with the McNemar test set up
#https://jameshoward.us/2024/12/17/mcnemars-test-the-hidden-gem-for-paired-binary-data

#determine what predictions for are our model are correct
ours_correct = (samples_few == ground_truth_few)
baseline_correct = (samples == ground_truth) 

#determine disagreements between models
ours_correct_baseline_incorrect = np.sum(ours_correct & ~baseline_correct)
ours_incorrect_baseline_correct = np.sum(~ours_correct & baseline_correct)

#create table for McNemars
#total correct for both, and number where ours was correct and baseline was incorrect/number where ours was incorrect and baseline was correct and total incorrect for both
table = [[np.sum(ours_correct & baseline_correct), ours_correct_baseline_incorrect],
         [ours_incorrect_baseline_correct, np.sum(~ours_correct & ~baseline_correct)]]

result = mcnemar(table, exact=True)
print(result)

#get hamming loss so we know the fraction of incorrect labels from misses and incorrect predictions for all the samples and classes
few_shot_hamming = hamming_loss(ground_truth_few, samples_few)
one_shot_hamming = hamming_loss(ground_truth, samples)

print(f"Few-Shot Hamming Loss: {few_shot_hamming}")
print(f"One-Shot Hamming Loss: {one_shot_hamming}")

In [ ]:
# This geeksforgeeks link helped with the line chart plot: https://www.geeksforgeeks.org/python/line-chart-in-matplotlib-python/
# Plot the line graphs for model throughput on non-thinking results

non_think_files = {
    'time_model_results_train_zero_no_think.csv': 'Zero-Shot, Training',
    'time_model_results_train_few_no_think.csv': 'Few-Shot, Training',
    'time_model_results_validate_zero_no_think.csv': 'Zero-Shot, Validate',
    'time_model_results_validate_few_no_think.csv': 'Few-Shot, Validate'
}

linestyles = ['-', '--', '-.', ':']
markers = ['o', 's', '^', 'D']
plt.figure(figsize=(10, 5))

for i, (file, label) in enumerate(non_think_files.items()):
    df = pd.read_csv(file)
    df['Cummulative Time'] = df['Total Model Time'].cumsum() / 60
    df['Cummulative Tokens'] = df['Total Tokens'].cumsum()
    plt.plot(df['Cummulative Tokens'], df['Cummulative Time'], label=label, linestyle=linestyles[i % len(linestyles)], marker=markers[i % len(markers)], markersize=5, linewidth=2, alpha=0.8)


plt.title('Non-Thinking Throughput: Cumulative Tokens vs. Time (Minutes)')
plt.xlabel('Cumulative Tokens')
plt.ylabel('Cumulative Time (minutes)')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()


plt.savefig('non_thinking_throughput.png', dpi=300)
plt.show() 

In [ ]:
# This geeksforgeeks link helped with the line chart plot: https://www.geeksforgeeks.org/python/line-chart-in-matplotlib-python/
# Plot the line graphs for model throughput on thinking results
think_files = {
    'time_model_results_train_zero_think.csv': 'Zero-Shot, Training',
    'time_model_results_train_few_think.csv': 'Few-Shot, Training',
    'time_model_results_validate_zero_think.csv': 'Zero-Shot, Validate',
    'time_model_results_validate_few_think.csv': 'Few-Shot, Validate',
    'time_model_results_test_zero_think.csv': 'Zero-Shot, Test',
    'time_model_results_test_few_think.csv': 'Few-Shot, Test'
}

linestyles = ['-', '--', '-.', ':', '-', '--']
markers = ['o', 's', '^', 'D', 'v', '*']
plt.figure(figsize=(10, 5))

for i, (file, label) in enumerate(think_files.items()):
    df = pd.read_csv(file)
    df['Cummulative Time'] = df['Total Model Time'].cumsum() / 60
    df['Cummulative Tokens'] = df['Total Tokens'].cumsum()
    plt.plot(df['Cummulative Tokens'], df['Cummulative Time'], label=label,  linestyle=linestyles[i % len(linestyles)], marker=markers[i % len(markers)], markersize=5, linewidth=2, alpha=0.8)


plt.title('Thinking Throughput: Cumulative Tokens vs. Time (Minutes)')
plt.xlabel('Cumulative Tokens')
plt.ylabel('Cumulative Time (minutes)')
plt.grid(True, alpha=0.6)
plt.legend()


plt.savefig('thinking_throughput.png', dpi=300)
plt.show() 